<a href="https://colab.research.google.com/github/davidriveraarbelaez/Optativa3_Agentes_IA/blob/main/Curso_agentes_inteligentes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Módulo 1



In [1]:
import random
import time

class VacuumAgent:
    def __init__(self):
        self.position = random.choice(["A", "B"])
        self.environment = {
            "A": random.choice(["clean", "dirty"]),
            "B": random.choice(["clean", "dirty"])
        }

    def perceive(self):
        return self.environment[self.position]

    def act(self):
        perception = self.perceive()

        if perception == "dirty":
            print(f" Limpiando en {self.position}")
            self.environment[self.position] = "clean"
        else:
            self.move()

    def move(self):
        self.position = "B" if self.position == "A" else "A"
        print(f" Moviéndose a {self.position}")

    def status(self):
        print("Entorno:", self.environment)

In [2]:
agent = VacuumAgent()

for step in range(6):
    print(f"\nPaso {step+1}")
    agent.status()
    agent.act()
    time.sleep(1)


Paso 1
Entorno: {'A': 'dirty', 'B': 'dirty'}
 Limpiando en B

Paso 2
Entorno: {'A': 'dirty', 'B': 'clean'}
 Moviéndose a A

Paso 3
Entorno: {'A': 'dirty', 'B': 'clean'}
 Limpiando en A

Paso 4
Entorno: {'A': 'clean', 'B': 'clean'}
 Moviéndose a B

Paso 5
Entorno: {'A': 'clean', 'B': 'clean'}
 Moviéndose a A

Paso 6
Entorno: {'A': 'clean', 'B': 'clean'}
 Moviéndose a B


#Módulo 2

In [3]:
%%capture
!pip install -q google-generativeai

In [4]:
import google.generativeai as genai

API_KEY = "MI_API_KEY_DESDE_GOOGLE_AI_STUDIO"
genai.configure(api_key=API_KEY)

model = genai.GenerativeModel("gemini-2.5-flash")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [5]:
system_prompt = """
Eres un agente supervisor energético.

Analiza datos de consumo y responde en JSON con:

{
 "status": "OK | ALERT",
 "analysis": "",
 "recommended_action": ""
}

No inventes datos.
"""

In [6]:
def energy_agent(current, average):

    user_input = f"Consumo actual {current}W, promedio {average}W"

    prompt = system_prompt + "\n" + user_input

    response = model.generate_content(prompt)

    return response.text

In [7]:
print(energy_agent(180, 120))

```json
{
 "status": "ALERT",
 "analysis": "El consumo actual de 180W es un 50% superior al consumo promedio registrado de 120W. Esto indica un incremento significativo que requiere atención.",
 "recommended_action": "Investigar la causa del aumento del consumo. Verificar si hay equipos adicionales en funcionamiento, si algún equipo está operando de manera ineficiente o si hay alguna anomalía en el sistema eléctrico. Se sugiere un monitoreo más detallado en las próximas horas."
}
```


In [8]:
memory = []

def energy_agent_with_memory(current, average):

    memory.append((current, average))

    context = "\n".join(
        [f"Histórico: {c}W vs {a}W" for c, a in memory[-3:]]
    )

    prompt = system_prompt + context + f"\nConsumo actual {current}W, promedio {average}W"

    response = model.generate_content(prompt)

    return response.text


#Módulo 3



In [9]:
class IntelligentAgent:

    def __init__(self, role):
        self.role = role
        self.memory = []

    def perceive(self, input_data):
        self.memory.append(input_data)

    def plan(self):
        return f"Plan basado en {len(self.memory)} observaciones"

    def act(self):
        return "Ejecutando acción"

    def run(self, input_data):
        self.perceive(input_data)
        plan = self.plan()
        action = self.act()

        return {
            "role": self.role,
            "plan": plan,
            "action": action
        }

In [10]:
agent = IntelligentAgent("Supervisor IoT")

result = agent.run("Consumo alto detectado")

print(result)

{'role': 'Supervisor IoT', 'plan': 'Plan basado en 1 observaciones', 'action': 'Ejecutando acción'}


In [11]:
class LLMAgent:

    def __init__(self, role):
        self.role = role
        self.memory = []

    def perceive(self, data):
        self.memory.append(data)

    def decide(self, data):

        context = "\n".join(self.memory[-3:])

        prompt = f"""
Eres {self.role}.
Histórico:
{context}

Analiza:
{data}
"""

        response = model.generate_content(prompt)

        return response.text

    def run(self, data):
        self.perceive(data)
        decision = self.decide(data)

        return decision

In [12]:
iot_agent = LLMAgent("Agente supervisor energético")

print(iot_agent.run("Consumo actual 220W, promedio 120W"))

¡Excelente! Como su Agente Supervisor Energético, procedo a analizar el histórico y el estado actual:

---

**Análisis de Consumo Energético**

**Datos Suministrados:**
*   **Consumo Actual:** 220W
*   **Consumo Promedio Histórico:** 120W

**Diagnóstico Inicial:**

La observación más crítica e inmediata es que el **consumo actual (220W) es significativamente superior al consumo promedio histórico (120W)**. Específicamente, el consumo actual representa un **incremento de aproximadamente el 83%** sobre el promedio, es decir, estamos consumiendo casi el doble de energía en este momento en comparación con nuestro patrón habitual.

**Implicaciones y Posibles Causas:**

1.  **Aumento Inesperado de Carga:**
    *   **Encendido de Equipo Adicional:** Es la causa más probable. Un equipo de alto consumo que no forma parte del patrón promedio (ej. un calefactor, aire acondicionado, horno, secadora de ropa, bomba de agua, herramientas eléctricas, etc.) ha sido encendido y está operando.
    *   **